In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()  # Load environment variables from .env file

gemini = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

gemini_model = "gemini-3.5-flash-lite"  # Change this to the desired Gemini model

ollama = OpenAI(
    api_key="ollama",  # Ollama does not require an API key by default
    base_url="http://localhost:11434/v1",
)

ollama_model = "gemma3:1b"  # Change this to the desired Ollama model

client = gemini  # Change this to `gemini` to use Gemini instead
model = gemini_model  # Change this to `gemini_model` to use Gemini instead

In [1]:
## Database mockup
import sqlite3

database_file = "marketplace.db"


def create_products_table():
    print("Creating products table if it doesn't exist...")
    conn = sqlite3.connect(database_file)
    with conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS products (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                category TEXT NOT NULL,
                description TEXT NOT NULL,
                price REAL NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
            """
        )

def insert_product(name, category, description, price):
    print(f"Inserting product: {name}, {category}, {description}, {price}")
    conn = sqlite3.connect(database_file)
    with conn:
        conn.execute(
            """
            INSERT INTO products (name, category, description, price)
            VALUES (?, ?, ?, ?)
            """,
            (name, category, description, price),
        )

samples = [
    {
        "name": "Wireless Earbuds",
        "category": "Electronics",
        "description": "High-quality wireless earbuds with noise cancellation.",
        "price": 99.99,
    },
    {
        "name": "Smartwatch",
        "category": "Wearables",
        "description": "A sleek smartwatch with fitness tracking features.",
        "price": 149.99,
    },
    {
        "name": "Gaming Laptop",
        "category": "Computers",
        "description": "A powerful gaming laptop with high-end graphics.",
        "price": 1299.99,
    },
]


create_products_table()
for sample in samples:
    insert_product(
        name=sample["name"],
        category=sample["category"],
        description=sample["description"],
        price=sample["price"]
    )

print("Sample products inserted into the database.")

Creating products table if it doesn't exist...
Inserting product: Wireless Earbuds, Electronics, High-quality wireless earbuds with noise cancellation., 99.99
Inserting product: Smartwatch, Wearables, A sleek smartwatch with fitness tracking features., 149.99
Inserting product: Gaming Laptop, Computers, A powerful gaming laptop with high-end graphics., 1299.99
Sample products inserted into the database.


In [5]:
SYSTEM_PROMPT = """
Você é um assistente útil que fornece informações sobre produtos em um mercado.
"""

def chat(message, history, model=model, client=client):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    response = client.chat.completions.create(
        model=model,
        messages= [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    )
    return response.choices[0].message.content

In [6]:
import gradio as gr

gr.ChatInterface(fn=chat, title="Marketplace Updater", description="Pergunte sobre os produtos do estoque.").launch()

d:\Dev\pocs-automacao\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
# Database query functions

def get_products_by_price_range(min_price, max_price):
    conn = sqlite3.connect(database_file)
    with conn:
        cursor = conn.execute(
            """
            SELECT * FROM products
            WHERE price BETWEEN ? AND ?
            """,
            (min_price, max_price),
        )
        return cursor.fetchall()

def get_products_by_category(category):
    conn = sqlite3.connect(database_file)
    with conn:
        cursor = conn.execute(
            """
            SELECT * FROM products
            WHERE category = ?
            """,
            (category,),
        )
        return cursor.fetchall()

def get_product_by_id(product_id):
    conn = sqlite3.connect(database_file)
    with conn:
        cursor = conn.execute(
            """
            SELECT * FROM products
            WHERE id = ?
            """,
            (product_id,),
        )
        return cursor.fetchone()

def get_all_products():
    conn = sqlite3.connect(database_file)
    with conn:
        cursor = conn.execute(
            """
            SELECT * FROM products
            """
        )
        return cursor.fetchall()

In [17]:
get_all_products()

[(1,
  'Wireless Earbuds',
  'Electronics',
  'High-quality wireless earbuds with noise cancellation.',
  99.99,
  '2026-09-25 22:41:12',
  '2026-09-25 22:41:12'),
 (2,
  'Smartwatch',
  'Wearables',
  'A sleek smartwatch with fitness tracking features.',
  149.99,
  '2026-09-25 22:41:12',
  '2026-09-25 22:41:12'),
 (3,
  'Gaming Laptop',
  'Computers',
  'A powerful gaming laptop with high-end graphics.',
  1299.99,
  '2026-09-25 22:41:12',
  '2026-09-25 22:41:12')]

In [9]:
# Tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_products_by_price_range",
            "description": "Retrieve products within a specified price range.",
            "parameters": {
                "type": "object",
                "properties": {
                    "min_price": {"type": "number", "description": "Minimum price"},
                    "max_price": {"type": "number", "description": "Maximum price"},
                },
                "required": ["min_price", "max_price"],
                "additionalProperties": False,
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_products_by_category",
            "description": "Retrieve products by category.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "description": "Product category"},
                },
                "required": ["category"],
                "additionalProperties": False,
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_product_by_id",
            "description": "Retrieve a product by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "integer", "description": "Product ID"},
                },
                "required": ["product_id"],
                "additionalProperties": False,
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_all_products",
            "description": "Retrieve all products in the database.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        }
    }
]

tools_dict = {
    "get_products_by_price_range": get_products_by_price_range,
    "get_products_by_category": get_products_by_category,
    "get_product_by_id": get_product_by_id,
    "get_all_products": get_all_products,
}

In [10]:
import json

def handle_tool_calls(message):
    print("Handling tool calls...")
    print("Message received for tool calls:", message)
    responses = []
    for toolcall in message.tool_calls:
        tool_name = toolcall.function.name
        tool_args = json.loads(toolcall.function.arguments)

        if tool_name not in tools_dict:
            responses.append(
            {
                "role": "tool",
                "tool_call_id": toolcall.id,
                "content": f"Tool '{tool_name}' not found.",
            }
            )
            continue

        tool_func = tools_dict[tool_name]
        tool_result = tool_func(**tool_args)
        responses.append(
            {
            "role": "tool",
            "tool_call_id": toolcall.id,
            "content": str(tool_result),
            }
        )

        return responses

def chat(message, history, model=model, client=client):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(tool_responses)

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [15]:
from IPython.display import display, Markdown

display(Markdown(chat("Quais produtos custam entre R$100 e R$500?", [])))

Handling tool calls...
Message received for tool calls: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_314984', function=Function(arguments='{"max_price":500,"min_price":100}', name='get_products_by_price_range'), type='function', extra_content={'google': {'thought_signature': 'EmAKXgFpFH0TCFG5BJQ2SAH0lMHOWe6RAGpdmVoJvA17cvsTi80wWvIuHS6Xx5pmUFeck5MCL2EPkg6qfyYrB5d3sJ57no2HOGTu36CxH8/E6xAz2nwKpXyyiQohnXHlzTs='}})])


O produto que custa entre R$100 e R$500 é:

* **Smartwatch** (Wearables) - R$149,99
  * *Descrição:* A sleek smartwatch with fitness tracking features.

In [ ]:
gr.ChatInterface(fn=chat, title="Marketplace Updater", description="Pergunte sobre os produtos do estoque.").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Handling tool calls...
Message received for tool calls: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_288484', function=Function(arguments='{"min_price":100,"max_price":500}', name='get_products_by_price_range'), type='function', extra_content={'google': {'thought_signature': 'EmAKXgFpFH0TEYLW9P4p+go71550so6kiS6g0eU5F7D8OkjPNLtK/fqSC3HLk55QkXsn1IXw2ro0u7Qyl35xlSRNq9rwyUEM6iUGft22SMdSA8GraoulvKrI4P6WVbcZZEc='}})])
Handling tool calls...
Message received for tool calls: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_356248', function=Function(arguments='{}', name='get_all_products'), type='function', extra_content={'google': {'thought_signature': 'EmAKXgFpFH0TfdIFijitkWctfq5KtBnRprSgU50pSF2KWPbQZb9aRiRgo62RJKmRAhw83sDhsvxfvYS7PzR42

In [ ]:
def update_product(product_id, name=None, category=None, description=None, price=None):
    conn = sqlite3.connect(database_file)
    with conn:
        product = get_product_by_id(product_id)
        if not product:
            return f"Product with ID {product_id} not found."

        updated_name = name if name is not None else product[1]
        updated_category = category if category is not None else product[2]
        updated_description = description if description is not None else product[3]
        updated_price = price if price is not None else product[4]

        conn.execute(
            """
            UPDATE products
            SET name = ?, category = ?, description = ?, price = ?, updated_at = CURRENT_TIMESTAMP
            WHERE id = ?
            """,
            (updated_name, updated_category, updated_description, updated_price, product_id),
        )

        conn.commit()

        # Return the updated product
        return get_product_by_id(product_id)


In [26]:
update_product(1, price=90.00)

(1,
 'Wireless Earbuds',
 'Electronics',
 'High-quality wireless earbuds with noise cancellation.',
 120.0,
 '2026-09-25 22:41:12',
 '2026-09-25 23:22:35')

In [27]:
get_product_by_id(1)

(1,
 'Wireless Earbuds',
 'Electronics',
 'High-quality wireless earbuds with noise cancellation.',
 90.0,
 '2026-09-25 22:41:12',
 '2026-09-25 23:22:56')

In [28]:
def update_product_tool(product_id, name=None, category=None, description=None, price=None):
    updated_product = update_product(product_id, name, category, description, price)
    return f"Product with ID {product_id} has been updated."

tools.append(
    {
        "type": "function",
        "function": {
            "name": "update_product_tool",
            "description": "Update a product's details by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {"type": "integer", "description": "Product ID"},
                    "name": {"type": "string", "description": "New product name"},
                    "category": {"type": "string", "description": "New product category"},
                    "description": {"type": "string", "description": "New product description"},
                    "price": {"type": "number", "description": "New product price"},
                },
                "required": ["product_id"],
                "additionalProperties": False,
            },
        }
    }
)

In [ ]:
display(Markdown(chat("Atualize o preço do produto com ID 1 para R$120,00.", [])))

In [29]:
gr.ChatInterface(fn=chat, title="Marketplace Updater", description="Pergunte sobre os produtos do estoque.").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Handling tool calls...
Message received for tool calls: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_269412', function=Function(arguments='{}', name='get_all_products'), type='function', extra_content={'google': {'thought_signature': 'EmAKXgFpFH0TQDNKiudKj1l7SqiRWzMae1lE1C8DwupUkaKHAxgZ7cCFIOhHkXcH0oHKWf18huiMBoHGCNgo6mb0yzDwY3ASvzwRxA05dLmPD3HL1aomMeY51jypI7nt8c8='}})])
Handling tool calls...
Message received for tool calls: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_212901', function=Function(arguments='{}', name='get_all_products'), type='function', extra_content={'google': {'thought_signature': 'EmAKXgFpFH0Tag/a93hGAyKLNGHNbN8jo9J7bHyrAQHLEXKmdZYnEe+VaoxCJYU6Vbt0PV8lsV+E5LV4UDsyRUKilXjzyRQacnM0EEqchfXOpGSnVD7NzqgpcP4ZD0O

Traceback (most recent call last):
  File "d:\Dev\pocs-automacao\.venv\Lib\site-packages\gradio\queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\pocs-automacao\.venv\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\pocs-automacao\.venv\Lib\site-packages\gradio\blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\pocs-automacao\.venv\Lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\pocs-automacao\.venv\Lib\site-packages\gradio\utils.py", line 1083, in async_wrapper
    response = await f(*args, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "